# Configure & Run Experiment

Fill in the fields below to build a config and run the full 4-step
pipeline (preprocess -> run model -> postprocess -> plot). Advanced/rarely-
changed settings are collapsed under "Advanced settings".

**Before you start**: set **Experiment root** to a directory you own
(not the instructor's data) — you write your own runs there, and
**Preprocess dir** to wherever the base climatology `.pt` files you want to
use already live (e.g. one generated for you, or one you built in the
other notebook here).

If **Cold start** is checked and the target experiment directory already
exists, you'll be asked to confirm before anything is deleted.


In [ ]:
import sys, os

def _find_project_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.exists(os.path.join(d, "scripts", "_config.py")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("Could not find the project root (looked for scripts/_config.py above "
                        + start + "). Make sure this notebook is somewhere inside the repo.")

PROJECT_ROOT = _find_project_root(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts"))
print("Project root:", PROJECT_ROOT)


In [ ]:
import datetime
import glob
import re
import subprocess
import ipywidgets as w
import yaml

from generate_heating import generate_heating_file
from generate_shape_scale import fit_gamma_shape_scale, zero_shape_scale

_LABEL_STYLE = {"description_width": "150px"}


def _wide():
    # A fresh Layout instance per widget -- widgets given the *same* Layout
    # object share that object's traits, so toggling .layout.display on one
    # would silently toggle it on every other widget using it too.
    return w.Layout(width="420px")


def _group_box():
    # Bordered/padded container so a generation panel's fields visually read
    # as belonging to its header, not as loose widgets floating below it.
    return w.Layout(border="1px solid #999", padding="8px", margin="4px 0px")


# Real, verified control experiments — every default in this notebook
# traces back to one of these two (config/experiments/T63L26_JJA_1979-2023.yaml,
# config/experiments/AC_Test.yaml). run_length_days/cold_start/toffset/spinup_days
# stay at short first-test values for both model types (AC_Test's real
# run_length_days is 54750 -- ~150 simulated years -- not a sane UI default).
FIXED_CONTROL = dict(
    season="JJA", heating_name="JJA_1979-2023", start_year=1979, end_year=2023,
    preprocess_path="/data/esplab/kpegion/projects/AGCM/MultiThread_Model/preprocess__zw_63__kmax_26",
)
# start_year/end_year=1994/2024 match what actually produced the real
# AC_Test/AC_warm/AC_noheating data (scripts/01_preprocess.py's
# gamma_preprocess_surface_pressure/_winds/_temperature slice reanalysis
# data to this range -- traced to Gamma_AC_Model/reference_notebooks/
# preprocess_gamma.ipynb, which they're a direct port of). AC_Test.yaml's
# own stated 1979/2023 was never actually used by anything -- that mismatch
# was only caught and fixed this session.
GAMMA_CONTROL = dict(
    heating_name="Test", start_year=1994, end_year=2024,
    preprocess_path="/data/esplab/kpegion/projects/AGCM/AnnualCycle",
)
# Instructor's own experiment_root (config/defaults.yaml) -- shown as a
# concrete, working suggestion. Students don't have write access there, so
# leaving it as-is just fails with a permission error; change it to your own.
SUGGESTED_EXPERIMENT_ROOT = "/data/esplab/kpegion/projects/AGCM_Experiments"

# --- Curated (main) fields ---
r_model_type = w.Dropdown(options=["fixed_season", "gamma_ac"], value="fixed_season", description="Model:", style=_LABEL_STYLE)
r_experiment_root = w.Text(value=SUGGESTED_EXPERIMENT_ROOT,
                            description="Experiment root:", placeholder="a directory you own -- replace the suggestion above",
                            style=_LABEL_STYLE, layout=_wide())
r_experiment_name = w.Text(description="Experiment name:", placeholder="directory name under experiment_root -- also names the saved config file", style=_LABEL_STYLE, layout=_wide())
r_season = w.Dropdown(options=["DJF", "JJA", "MAM", "SON"], value=FIXED_CONTROL["season"], description="Season:", style=_LABEL_STYLE)
# r_heating_name lives inside heating_gen_box/ss_gen_box below (it's part of
# "what am I generating", not a top-level run setting) -- defined here since
# many other widgets below reference it, but never placed at the top level.
# Its value is entirely auto-generated from whichever Heating/Shape-scale
# source is selected below (see on_heating_gen_mode_change/on_ss_gen_mode_change
# and the per-file-field observers further down) -- still editable afterward,
# same as every other auto-filled field in this notebook.
r_heating_name = w.Text(value=FIXED_CONTROL["heating_name"], description="Heating name:", style=_LABEL_STYLE, layout=_wide())
r_start_year = w.IntText(value=FIXED_CONTROL["start_year"], description="Start yr:", style=_LABEL_STYLE)
r_end_year = w.IntText(value=FIXED_CONTROL["end_year"], description="End yr:", style=_LABEL_STYLE)
r_preprocess_path = w.Text(value=FIXED_CONTROL["preprocess_path"],
                            description="Preprocess dir:", placeholder="path to base climatology .pt files", style=_LABEL_STYLE, layout=_wide())
r_run_length_days = w.IntText(value=30, description="Run length (days):", style=_LABEL_STYLE)
r_cold_start = w.Checkbox(value=True, description="Cold start", style=_LABEL_STYLE)
r_toffset = w.IntText(value=0, description="Restart offset (days):", style=_LABEL_STYLE)
r_shape_file = w.Text(description="Shape file:", placeholder="(gamma_ac) blank = control default shapeAC.pt", style=_LABEL_STYLE, layout=_wide())
r_scale_file = w.Text(description="Scale file:", placeholder="(gamma_ac) blank = control default scaleAC.pt", style=_LABEL_STYLE, layout=_wide())
r_control_experiment = w.Text(description="Control exp:", placeholder="(optional) name of a control run to diff against", style=_LABEL_STYLE, layout=_wide())
r_spinup_days = w.IntText(value=60, description="Spinup days:", style=_LABEL_STYLE)
# Drives both step 3 (postprocess -- interpolate to pressure levels) and
# step 4 (plot) with the same variable list. Previously two separate fields
# (Postprocess vars / Plot vars) that could disagree -- e.g. listing a var
# in Plot vars that wasn't in Postprocess vars would fail at step 4 since
# the file it needs was never written. One field removes that failure mode;
# defaults to all four available variables.
r_plot_vars = w.SelectMultiple(options=["uvel", "vvel", "geo", "temp"], value=["uvel", "vvel", "geo", "temp"],
                                description="Vars (postprocess &amp; plot):", style=_LABEL_STYLE)

# Hover-tooltip info icon (native browser tooltip via the HTML title attribute
# -- no extra click, no extra state) explaining what Start/End yr mean, which
# differs by model: load-bearing for fixed_season, currently just a label for
# gamma_ac's diagnostic file but does drive its real climatology period too
# (see scripts/01_preprocess.py's gamma_preprocess_temperature/_surface_pressure/
# _winds, which now read these instead of a hardcoded range).
r_years_info = w.HTML()


def _update_years_info(*_):
    if r_model_type.value == "fixed_season":
        tip = ("Climatology period actually used: NCEP reanalysis fields are averaged over this "
               "range, then combined into the chosen Season&#39;s mean -- this directly determines "
               "the model&#39;s background climate.")
    else:
        tip = ("Climatology period used to build gamma_ac&#39;s annual-cycle background state "
               "(temperature, surface pressure, winds) from NCEP reanalysis.")
    r_years_info.value = f"<span title=\"{tip}\" style='cursor:help;color:#666;'>&#9432; what do these mean?</span>"


r_model_type.observe(_update_years_info, names="value")
_update_years_info()


def experiment_dir():
    return os.path.join(r_experiment_root.value, r_experiment_name.value)


# --- Restart offset auto-detection ---
# Both model runners write every output chunk as {var}_{start}_{end}.nc with
# dates anchored to this same fixed epoch (scripts/02_run_model.py:146,
# Gamma_AC_Model/RunModel.Gamma.py:106 -- both pd.date_range(start="1950-01-01", ...)),
# independent of the experiment's own start_year/end_year. toffset is just
# "days since this epoch" (02_run_model.py's own docstring: "toffset=<days
# already run>"), so it can be derived directly from what's already on disk
# instead of hand-computed.
_MODEL_EPOCH = datetime.date(1950, 1, 1)
_UVEL_CHUNK_RE = re.compile(r"^uvel_\d{4}-\d{2}-\d{2}_(\d{4}-\d{2}-\d{2})\.nc$")


def _days_already_run(exp_dir):
    end_dates = []
    for f in glob.glob(os.path.join(exp_dir, "uvel_*.nc")):
        m = _UVEL_CHUNK_RE.match(os.path.basename(f))
        if m:
            end_dates.append(datetime.date.fromisoformat(m.group(1)))
    if not end_dates:
        return None
    return (max(end_dates) - _MODEL_EPOCH).days + 1


r_toffset_status = w.HTML()


def _update_toffset_status(*_):
    if r_cold_start.value:
        r_toffset.value = 0
        r_toffset_status.value = "<i>Cold start — Restart offset is 0.</i>"
        return
    exp_dir = experiment_dir()
    days = _days_already_run(exp_dir)
    if days is None:
        r_toffset_status.value = (f"<span style='color:#b30000'>No existing output found in "
                                   f"<code>{exp_dir}</code>.</span> Set Restart offset manually, "
                                   "or check Cold start to begin fresh.")
    else:
        r_toffset_status.value = (f"<span style='color:green'>Detected {days} day(s) already run in "
                                   f"<code>{exp_dir}</code>.</span> Restart offset set to match "
                                   "(edit it yourself if you want something else).")
        r_toffset.value = days


r_cold_start.observe(_update_toffset_status, names="value")
r_experiment_root.observe(_update_toffset_status, names="value")
r_experiment_name.observe(_update_toffset_status, names="value")
_update_toffset_status()

# --- Heating generator (fixed_season) ---
HEATING_GEN_MODES = ["Use control default (JJA 1979-2023)", "Generate new: custom file",
                      "Generate new: from CCA precip", "Generate new: from CESM2 precip",
                      "Generate new: from ERA5 precip"]
_HEATING_SOURCE_BY_MODE = {
    HEATING_GEN_MODES[0]: "custom", HEATING_GEN_MODES[1]: "custom",
    HEATING_GEN_MODES[2]: "cca", HEATING_GEN_MODES[3]: "cesm2", HEATING_GEN_MODES[4]: "era5",
}
r_heating_gen_mode = w.Dropdown(options=HEATING_GEN_MODES, value=HEATING_GEN_MODES[0],
                                 description="Heating source:", style=_LABEL_STYLE, layout=_wide())
r_heating_reset_note = w.HTML(
    "<i>Heating name (below) auto-fills from whichever source is picked here -- edit it "
    "yourself if you want something else. \"Use control default\" also resets Season / "
    "Start-End yr / Preprocess dir to the control values.</i>")
r_heating_file = w.Text(description="Heating file:", placeholder="path to a pre-made heat.ggrid .pt file to copy in -- Heating name above will auto-fill from it", style=_LABEL_STYLE, layout=_wide())
r_cesm2_precip_file = w.Text(description="CESM2 precip file:", style=_LABEL_STYLE, layout=_wide())
r_cca_precip_file = w.Text(description="CCA precip file:", style=_LABEL_STYLE, layout=_wide())
r_era5_precip_file = w.Text(description="ERA5 precip file:", style=_LABEL_STYLE, layout=_wide())
heating_gen_output = w.Output()
heating_gen_button = w.Button(description="Generate Heating File", button_style="warning")


def _derive_heating_name_from_file(path):
    # Strip extension and the heat.ggrid_/heat_/.ggrid naming-convention
    # noise seen in real files (e.g. heat_DJF_1999-2020_ALL.ggrid.pt ->
    # DJF_1999-2020_ALL) so the auto-filled label is clean, not a literal
    # dump of the source filename.
    base = os.path.splitext(os.path.basename(path))[0]
    for prefix in ("heat.ggrid_", "heat_"):
        if base.startswith(prefix):
            base = base[len(prefix):]
            break
    if base.endswith(".ggrid"):
        base = base[:-len(".ggrid")]
    return base


def _make_heating_name_deriver(expected_mode):
    # One factory shared by the custom-file field and the three precip-file
    # fields: whichever field belongs to the currently-selected Heating
    # source re-derives Heating name when it changes. Heating name and a
    # second, separately-typed source field would otherwise read as
    # duplicative -- this keeps it to one thing to type per source.
    def handler(change):
        if r_heating_gen_mode.value == expected_mode and change["new"]:
            r_heating_name.value = _derive_heating_name_from_file(change["new"])
    return handler


r_heating_file.observe(_make_heating_name_deriver(HEATING_GEN_MODES[1]), names="value")
r_cca_precip_file.observe(_make_heating_name_deriver(HEATING_GEN_MODES[2]), names="value")
r_cesm2_precip_file.observe(_make_heating_name_deriver(HEATING_GEN_MODES[3]), names="value")
r_era5_precip_file.observe(_make_heating_name_deriver(HEATING_GEN_MODES[4]), names="value")

heating_status = w.HTML()


def _heating_target_path():
    return os.path.join(r_preprocess_path.value, f"heat.ggrid_{r_heating_name.value}.pt")


def _update_heating_status(*_):
    p = _heating_target_path()
    if os.path.exists(p):
        heating_status.value = (f"<span style='color:green'>&#10003; Heating file exists:</span> "
                                 f"<code>{p}</code> — Run Pipeline will use this as-is.")
    else:
        heating_status.value = (f"<span style='color:#b30000'>&#10007; Heating file NOT found:</span> "
                                 f"<code>{p}</code> — click Generate above, or Run Pipeline will refuse to start.")


r_heating_name.observe(_update_heating_status, names="value")
r_preprocess_path.observe(_update_heating_status, names="value")

# --- Shape/scale generator (gamma_ac) ---
SS_GEN_MODES = ["Use control default (shapeAC.pt / scaleAC.pt)", "Fit new: Control period",
                 "Fit new: Composite (e.g. El Nino)", "Generate: No heating (zero)"]
# Fixed labels for the fit-based modes: there's no source file to derive a
# name from (they compute new data from a date range, not copy a file), and
# a stable default matches this notebook's "generate once, then Run Pipeline
# against exactly that" flow better than baking in dates that would make the
# name drift every time a date field is tweaked. Still editable afterward.
_SS_MODE_FALLBACK_NAME = {
    SS_GEN_MODES[1]: "ControlFit", SS_GEN_MODES[2]: "Composite", SS_GEN_MODES[3]: "NoHeating",
}
r_ss_gen_mode = w.Dropdown(options=SS_GEN_MODES, value=SS_GEN_MODES[0],
                            description="Shape/scale source:", style=_LABEL_STYLE, layout=_wide())
r_ss_reset_note = w.HTML(
    "<i>Heating name (below) auto-fills from whichever source is picked here -- edit it "
    "yourself if you want something else. \"Use control default\" also resets Start-End yr / "
    "Preprocess dir to the control values, and clears Shape/Scale file below.</i>")
ss_precip_glob = w.Text(
    value="/data/esplab/shared/obs/gridded/atm/precip/daily/CMORPH/CMORPH_V1.0_ADJ_0.25deg-DLY_00Z_*.nc",
    description="Precip glob:", style=_LABEL_STYLE, layout=_wide())
ss_precip_varname = w.Text(value="cmorph", description="Precip var name:", style=_LABEL_STYLE)
ss_start_date = w.DatePicker(description="Fit start:", style=_LABEL_STYLE)
ss_end_date = w.DatePicker(description="Fit end:", style=_LABEL_STYLE)
ss_scale_qc_max = w.FloatText(value=300.0, description="Scale QC max:", style=_LABEL_STYLE)
ss_windows_box = w.VBox([])
ss_add_window_btn = w.Button(description="+ Add composite window")
ss_gen_output = w.Output()
ss_gen_button = w.Button(description="Generate Shape/Scale Files", button_style="warning")

ss_status = w.HTML()


def _ss_target_paths():
    shape_name = r_shape_file.value or "shapeAC.pt"
    scale_name = r_scale_file.value or "scaleAC.pt"
    return (os.path.join(r_preprocess_path.value, shape_name),
            os.path.join(r_preprocess_path.value, scale_name))


def _update_ss_status(*_):
    shape_path, scale_path = _ss_target_paths()
    parts = []
    for label, p, field in (("Shape", shape_path, r_shape_file), ("Scale", scale_path, r_scale_file)):
        tag = " (control default)" if not field.value else ""
        if os.path.exists(p):
            parts.append(f"<span style='color:green'>&#10003; {label} exists{tag}:</span> <code>{p}</code>")
        else:
            parts.append(f"<span style='color:#b30000'>&#10007; {label} NOT found{tag}:</span> <code>{p}</code>")
    ss_status.value = "<br>".join(parts)


r_shape_file.observe(_update_ss_status, names="value")
r_scale_file.observe(_update_ss_status, names="value")
r_preprocess_path.observe(_update_ss_status, names="value")


def make_window_row():
    s = w.DatePicker(description="Window start:", style=_LABEL_STYLE)
    e = w.DatePicker(description="Window end:", style=_LABEL_STYLE)
    rm = w.Button(description="Remove", button_style="danger", layout=w.Layout(width="80px"))
    row = w.HBox([s, e, rm])

    def on_remove(b):
        ss_windows_box.children = tuple(c for c in ss_windows_box.children if c is not row)

    rm.on_click(on_remove)
    return row


def on_add_window(b):
    ss_windows_box.children = ss_windows_box.children + (make_window_row(),)


ss_add_window_btn.on_click(on_add_window)

# --- Advanced fields (collapsed) ---
a_zw = w.Dropdown(options=[42, 63, 124], value=63, description="zw:", style=_LABEL_STYLE)
a_kmax = w.Dropdown(options=[11, 26], value=26, description="kmax:", style=_LABEL_STYLE)
a_chunk_size_days = w.IntText(value=30, description="Chunk size (days):", style=_LABEL_STYLE)
a_compute_slp = w.Checkbox(value=False, description="Compute SLP", style=_LABEL_STYLE)
a_heating_file_override = w.Text(description="Heating filename override:",
                                  placeholder="only if an existing file's name doesn't match heat.ggrid_{heating_name}.pt",
                                  style=_LABEL_STYLE, layout=_wide())
a_gamma_heating_custom_file = w.Text(description="Diagnostic heat.ggrid file:",
                                      placeholder="rare, gamma_ac only -- NOT used by the running model, purely diagnostic",
                                      style=_LABEL_STYLE, layout=_wide())

advanced_box = w.Accordion(children=[w.VBox([a_zw, a_kmax, a_chunk_size_days, a_compute_slp,
                                              a_heating_file_override, a_gamma_heating_custom_file])])
advanced_box.set_title(0, "Advanced settings (usually leave as default)")
advanced_box.selected_index = None  # collapsed by default

config_output = w.Output()
build_config_button = w.Button(description="Build Config", button_style="info")

run_output = w.Output()
run_button = w.Button(description="Run Pipeline (steps 1-4)", button_style="primary")
confirm_box = w.VBox([])  # populated with a confirmation button when needed


def on_heating_gen_mode_change(change):
    mode = change["new"]
    r_heating_file.layout.display = "" if mode == "Generate new: custom file" else "none"
    r_cca_precip_file.layout.display = "" if mode == "Generate new: from CCA precip" else "none"
    r_cesm2_precip_file.layout.display = "" if mode == "Generate new: from CESM2 precip" else "none"
    r_era5_precip_file.layout.display = "" if mode == "Generate new: from ERA5 precip" else "none"
    is_default = mode == HEATING_GEN_MODES[0]
    heating_gen_button.layout.display = "none" if is_default else ""
    if is_default:
        r_heating_name.value = FIXED_CONTROL["heating_name"]
        r_season.value = FIXED_CONTROL["season"]
        r_start_year.value = FIXED_CONTROL["start_year"]
        r_end_year.value = FIXED_CONTROL["end_year"]
        r_preprocess_path.value = FIXED_CONTROL["preprocess_path"]
    elif mode == HEATING_GEN_MODES[1]:
        r_heating_name.value = (_derive_heating_name_from_file(r_heating_file.value)
                                 if r_heating_file.value else "Custom")
    elif mode == HEATING_GEN_MODES[2]:
        r_heating_name.value = (_derive_heating_name_from_file(r_cca_precip_file.value)
                                 if r_cca_precip_file.value else "CCA")
    elif mode == HEATING_GEN_MODES[3]:
        r_heating_name.value = (_derive_heating_name_from_file(r_cesm2_precip_file.value)
                                 if r_cesm2_precip_file.value else "CESM2")
    elif mode == HEATING_GEN_MODES[4]:
        r_heating_name.value = (_derive_heating_name_from_file(r_era5_precip_file.value)
                                 if r_era5_precip_file.value else "ERA5")


r_heating_gen_mode.observe(on_heating_gen_mode_change, names="value")
on_heating_gen_mode_change({"new": r_heating_gen_mode.value})


def on_ss_gen_mode_change(change):
    mode = change["new"]
    is_control_fit = mode == "Fit new: Control period"
    is_composite = mode == "Fit new: Composite (e.g. El Nino)"
    is_default = mode == SS_GEN_MODES[0]
    for widget in (ss_precip_glob, ss_precip_varname, ss_start_date, ss_end_date):
        widget.layout.display = "" if (is_control_fit or is_composite) else "none"
    ss_scale_qc_max.layout.display = "" if is_composite else "none"
    ss_windows_box.layout.display = "" if is_composite else "none"
    ss_add_window_btn.layout.display = "" if is_composite else "none"
    ss_gen_button.layout.display = "none" if is_default else ""
    if is_default:
        r_heating_name.value = GAMMA_CONTROL["heating_name"]
        r_start_year.value = GAMMA_CONTROL["start_year"]
        r_end_year.value = GAMMA_CONTROL["end_year"]
        r_preprocess_path.value = GAMMA_CONTROL["preprocess_path"]
        r_shape_file.value = ""
        r_scale_file.value = ""
    else:
        r_heating_name.value = _SS_MODE_FALLBACK_NAME[mode]


r_ss_gen_mode.observe(on_ss_gen_mode_change, names="value")
on_ss_gen_mode_change({"new": r_ss_gen_mode.value})


def on_model_type_change(change):
    is_fixed = change["new"] == "fixed_season"
    r_season.layout.display = "" if is_fixed else "none"
    r_shape_file.layout.display = "none" if is_fixed else ""
    r_scale_file.layout.display = "none" if is_fixed else ""
    heating_gen_box.layout.display = "" if is_fixed else "none"
    ss_gen_box.layout.display = "none" if is_fixed else ""
    if is_fixed:
        r_heating_gen_mode.value = HEATING_GEN_MODES[0]
        on_heating_gen_mode_change({"new": HEATING_GEN_MODES[0]})
    else:
        r_ss_gen_mode.value = SS_GEN_MODES[0]
        on_ss_gen_mode_change({"new": SS_GEN_MODES[0]})


r_model_type.observe(on_model_type_change, names="value")


def on_generate_heating_clicked(b):
    with heating_gen_output:
        heating_gen_output.clear_output()
        try:
            kwargs = dict(
                model_type="fixed_season",
                heating_source=_HEATING_SOURCE_BY_MODE[r_heating_gen_mode.value],
                heating_name=r_heating_name.value,
                output_dir=r_preprocess_path.value,
                season=r_season.value,
                start_year=r_start_year.value,
                end_year=r_end_year.value,
                zw=a_zw.value,
                kmax=a_kmax.value,
            )
            mode = r_heating_gen_mode.value
            if mode == "Generate new: custom file":
                kwargs["heating_file"] = r_heating_file.value
            elif mode == "Generate new: from CCA precip":
                kwargs["cca_precip_file"] = r_cca_precip_file.value
            elif mode == "Generate new: from CESM2 precip":
                kwargs["cesm2_precip_file"] = r_cesm2_precip_file.value
            elif mode == "Generate new: from ERA5 precip":
                kwargs["era5_precip_file"] = r_era5_precip_file.value
            outfile = generate_heating_file(**kwargs)
            print(f"Done. Wrote: {outfile}")
        except Exception as e:
            print(f"ERROR: {e}")
        finally:
            _update_heating_status()


heating_gen_button.on_click(on_generate_heating_clicked)


def on_generate_ss_clicked(b):
    with ss_gen_output:
        ss_gen_output.clear_output()
        try:
            if not r_heating_name.value:
                raise ValueError("Heating name is required (used as the shape/scale file label).")
            if not r_preprocess_path.value:
                raise ValueError("Preprocess dir is required (shape/scale files are written there).")

            mode = r_ss_gen_mode.value
            if mode == "Generate: No heating (zero)":
                zero_shape_scale(r_preprocess_path.value, r_heating_name.value, a_zw.value)
            else:
                if not (ss_start_date.value and ss_end_date.value):
                    raise ValueError("Fit start/end date required.")
                windows = None
                if mode == "Fit new: Composite (e.g. El Nino)":
                    windows = []
                    for row in ss_windows_box.children:
                        s_widget, e_widget, _ = row.children
                        if not (s_widget.value and e_widget.value):
                            raise ValueError("Every composite window needs both a start and end date.")
                        windows.append((str(s_widget.value), str(e_widget.value)))
                    if not windows:
                        raise ValueError("Add at least one composite window, or switch to 'Fit new: Control period'.")
                fit_gamma_shape_scale(
                    precip_glob=ss_precip_glob.value,
                    precip_varname=ss_precip_varname.value,
                    date_range=(str(ss_start_date.value), str(ss_end_date.value)),
                    zw=a_zw.value,
                    output_dir=r_preprocess_path.value,
                    name=r_heating_name.value,
                    composite_windows=windows,
                    scale_qc_max=ss_scale_qc_max.value if mode == "Fit new: Composite (e.g. El Nino)" else None,
                )
            r_shape_file.value = f"shape_{r_heating_name.value}.pt"
            r_scale_file.value = f"scale_{r_heating_name.value}.pt"
            print(f"Done. Wrote shape_{r_heating_name.value}.pt / scale_{r_heating_name.value}.pt "
                  f"to {r_preprocess_path.value}")
        except Exception as e:
            print(f"ERROR: {e}")
        finally:
            _update_ss_status()


ss_gen_button.on_click(on_generate_ss_clicked)

heating_gen_box = w.VBox(
    [
        w.HTML("<b>Generate Heating File</b> (fixed_season) — the options below all configure this:"),
        r_heating_gen_mode, r_heating_reset_note,
        r_heating_name, heating_status,
        r_heating_file, r_cca_precip_file, r_cesm2_precip_file, r_era5_precip_file,
        heating_gen_button, heating_gen_output,
    ],
    layout=_group_box(),
)

ss_gen_box = w.VBox(
    [
        w.HTML("<b>Generate Shape/Scale Files</b> (gamma_ac — what the model actually uses) — "
               "the options below all configure this:"),
        r_ss_gen_mode, r_ss_reset_note,
        r_heating_name, ss_status,
        ss_precip_glob, ss_precip_varname, ss_start_date, ss_end_date, ss_scale_qc_max,
        ss_windows_box, ss_add_window_btn,
        ss_gen_button, ss_gen_output,
    ],
    layout=_group_box(),
)

postprocessing_box = w.VBox(
    [
        w.HTML("<b>Post-processing &amp; Plotting</b> — only used after the run finishes, to build figures:"),
        r_spinup_days, r_control_experiment, r_plot_vars,
    ],
    layout=_group_box(),
)

on_model_type_change({"new": r_model_type.value})
_update_heating_status()
_update_ss_status()


def build_cfg_dict():
    if not r_heating_name.value:
        raise ValueError("Heating name is required.")
    if not r_preprocess_path.value:
        raise ValueError("Preprocess dir is required.")
    if not r_experiment_root.value:
        raise ValueError("Experiment root is required (use your own directory).")
    if not r_experiment_name.value:
        raise ValueError("Experiment name is required.")

    cfg = {
        "model_type": r_model_type.value,
        "start_year": r_start_year.value,
        "end_year": r_end_year.value,
        "season": r_season.value if r_model_type.value == "fixed_season" else "annual",
        "heating_name": r_heating_name.value,
        "preprocess_path_override": r_preprocess_path.value,
        "experiment_root": r_experiment_root.value,
        "experiment_name": r_experiment_name.value,
        "run_length_days": r_run_length_days.value,
        "cold_start": r_cold_start.value,
        "toffset": r_toffset.value,
        "control_experiment": r_control_experiment.value or None,
        "spinup_days": r_spinup_days.value,
        # Same list drives both step 3's pressure-level interpolation and
        # step 4's plotting -- see r_plot_vars's definition above for why.
        "plot_vars": list(r_plot_vars.value),
        "postprocess_vars": list(r_plot_vars.value),
        "zw": a_zw.value,
        "kmax": a_kmax.value,
        "chunk_size_days": a_chunk_size_days.value,
        "compute_slp": a_compute_slp.value,
    }
    if r_model_type.value == "fixed_season":
        cfg["model_subtype"] = "weakly_prescribed_mean"
        cfg["heating_source"] = _HEATING_SOURCE_BY_MODE[r_heating_gen_mode.value]
        # heating_file is what step 1 (mt_preprocess_heating) reads for
        # heating_source=custom -- the raw file to copy in when generating a
        # new heating file. heating_file_override is a separate, unrelated
        # thing: a step-2-only filename override for when an existing file
        # on disk doesn't match the heat.ggrid_{heating_name}.pt formula.
        if r_heating_file.value:
            cfg["heating_file"] = r_heating_file.value
        if r_cca_precip_file.value:
            cfg["cca_precip_file"] = r_cca_precip_file.value
        if r_cesm2_precip_file.value:
            cfg["cesm2_precip_file"] = r_cesm2_precip_file.value
        if r_era5_precip_file.value:
            cfg["era5_precip_file"] = r_era5_precip_file.value
        if a_heating_file_override.value:
            cfg["heating_file_override"] = a_heating_file_override.value
    else:
        if r_shape_file.value:
            cfg["shape_file_override"] = r_shape_file.value
        if r_scale_file.value:
            cfg["scale_file_override"] = r_scale_file.value
        # heat.ggrid_{heating_name}.pt for gamma_ac is a diagnostic artifact
        # only (never loaded by RunModel.Gamma.py) and is never required for
        # preprocess completeness, so this only matters if a full (non
        # --heating-only) preprocess ever has to run from scratch.
        if a_gamma_heating_custom_file.value:
            cfg["heating_source"] = "custom"
            cfg["heating_file"] = a_gamma_heating_custom_file.value
        else:
            cfg["heating_source"] = "cmap_default"
    return cfg


def config_path():
    # Experiment name drives the saved config filename directly, so there's
    # no separate "Save config to" field that can drift out of sync with it.
    return f"{r_experiment_name.value}.yaml"


def on_build_config_clicked(b):
    with config_output:
        config_output.clear_output()
        try:
            cfg = build_cfg_dict()
            path = config_path()
            with open(path, "w") as f:
                yaml.safe_dump(cfg, f, sort_keys=False)
            print(f"Wrote config to {path}")
            print(yaml.safe_dump(cfg, sort_keys=False))
        except Exception as e:
            print(f"ERROR: {e}")


build_config_button.on_click(on_build_config_clicked)


def _check_inputs_ready():
    # No input file is ever generated as a side effect of Run Pipeline --
    # either the required file(s) already exist, or Run Pipeline refuses to
    # start and says exactly what's missing and what to click.
    problems = []
    if r_model_type.value == "fixed_season":
        p = _heating_target_path()
        if not os.path.exists(p):
            problems.append(f"Heating file not found: {p} -- click 'Generate Heating File' above, "
                             "or fix Heating name / Preprocess dir.")
    else:
        for label, p in zip(("Shape file", "Scale file"), _ss_target_paths()):
            if not os.path.exists(p):
                problems.append(f"{label} not found: {p} -- click 'Generate Shape/Scale Files' above, "
                                 "or fix the filename / Preprocess dir.")
    return problems


def _run_pipeline():
    with run_output:
        run_output.clear_output()
        path = config_path()
        if not os.path.exists(path):
            print("ERROR: build the config first.")
            return
        steps = ["01_preprocess.py", "02_run_model.py", "03_postprocess.py", "04_plot_results.py"]
        for step in steps:
            cmd = [sys.executable, os.path.join(PROJECT_ROOT, "scripts", step),
                   "--config", os.path.abspath(path)]
            print(f"--- Running {step} ---")
            result = subprocess.run(cmd, cwd=os.path.join(PROJECT_ROOT, "scripts"),
                                     capture_output=True, text=True)
            print(result.stdout)
            if result.returncode != 0:
                print(result.stderr)
                print(f"FAILED at {step} (exit {result.returncode}) — stopping.")
                return
        print("Pipeline complete.")


def on_run_clicked(b):
    with run_output:
        run_output.clear_output()
        problems = _check_inputs_ready()
        if problems:
            print("Cannot run -- required input file(s) missing:")
            for msg in problems:
                print(" -", msg)
            print("Nothing has been run.")
            return
    target = experiment_dir()
    if r_cold_start.value and os.path.isdir(target):
        confirm_btn = w.Button(description=f"Confirm: delete and restart {target}", button_style="danger")

        def on_confirm(cb):
            confirm_box.children = ()
            _run_pipeline()

        confirm_btn.on_click(on_confirm)
        confirm_box.children = (confirm_btn,)
        with run_output:
            print(f"cold_start=True and {target} already exists — click above to confirm overwrite.")
    else:
        _run_pipeline()


run_button.on_click(on_run_clicked)

run_panel = w.VBox([
    w.HTML("<b>Configure &amp; Run Experiment</b>"),
    r_model_type, r_experiment_root, r_experiment_name,
    r_season, r_start_year, r_end_year, r_years_info, r_preprocess_path,
    r_run_length_days, r_cold_start, r_toffset, r_toffset_status,
    heating_gen_box, ss_gen_box, r_shape_file, r_scale_file,
    postprocessing_box,
    advanced_box,
    build_config_button, config_output,
    run_button, confirm_box, run_output,
])

display(run_panel)
